# OpenAI Decisions API + Strands: two ways to use it

A decision model answers a bounded question: given a request and a fixed list of
options, it returns one of them with a confidence — no generated text, no option
it invents, nothing to parse. The **OpenAI Decisions API** (`gpt-6-luna`) is one.

This standalone notebook uses it inside a [Strands](https://strandsagents.com)
agent two ways, OpenAI only:

1. **Choose the tool** — a hook narrows a 40-tool pool to the one the request
   needs, before the model runs.
2. **Choose the model** — `ModelRouter` sends each request to the right-sized
   model.

**Setup:** a running Neo4j with the hotel graph (demo 01's `build_graph.py`) for
the tool-selection section; AWS creds for the Bedrock models; `OPENAI_API_KEY`
and `pip install "openai>=3.26.0"`.

In [ ]:
import os
os.environ.setdefault("OTEL_SDK_DISABLED", "true")
os.environ.setdefault("AWS_REGION", "us-east-1")
from dotenv import load_dotenv
load_dotenv()

from openai import AsyncOpenAI
OAI = AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])

def latest_user_text(messages):
    for m in reversed(messages):
        if m.get("role") == "user":
            return "".join(b.get("text", "") for b in m.get("content", []) if "text" in b)
    return ""

## Option 1: choose the tool

Give a small model 40 overlapping tools and it picks the wrong one or invents
one. A `BeforeModelCallEvent` hook asks the Decisions API which tool fits, then
swaps the agent's tool registry down to that one before the model runs. The
selector is one async function; the rest is the hook.

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from strands.hooks import HookProvider, HookRegistry, BeforeModelCallEvent
from semantic_tools import ALL_TOOLS

BY_NAME = {t.tool_name: t for t in ALL_TOOLS}
TOOL_CHOICES = [{"value": t.tool_name, "description": t.tool_spec["description"]} for t in ALL_TOOLS]

async def openai_select_tool(query):
    resp = await OAI.decisions.create(
        model="gpt-6-luna", input=query,
        questions=[{"type": "choice", "name": "tool",
                    "instructions": "Which tool answers this request?", "choices": TOOL_CHOICES}],
    )
    answer = next(a for a in resp.answers if getattr(a, "name", None) == "tool")
    return BY_NAME[answer.choice], answer.confidence

class OpenAIToolHook(HookProvider):
    def __init__(self):
        self.last = None
    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeModelCallEvent, self._before_model)
    async def _before_model(self, event: BeforeModelCallEvent) -> None:
        q = latest_user_text(event.agent.messages)
        if not q:
            return
        tool, conf = await openai_select_tool(q)
        self.last = (tool.tool_name, conf)
        reg = event.agent.tool_registry
        reg.registry.clear(); reg.dynamic_tools.clear()
        reg.register_tool(tool)

AGENT_MODEL = BedrockModel(model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0", region_name=os.environ["AWS_REGION"])

# --- Alternative: OpenAI (instead of the Bedrock model above) ---
# Install the provider: pip install 'strands-agents[openai]'
# (also listed, commented, in requirements.txt). Set OPENAI_API_KEY in .env.
# from strands.models.openai import OpenAIModel
# AGENT_MODEL = OpenAIModel(client_args={"api_key": os.environ["OPENAI_API_KEY"]}, model_id="gpt-4o-mini")
PROMPT = "You are a travel agent that helps people with hotels. Keep answers concise — at most two sentences."

hook = OpenAIToolHook()
tool_agent = Agent(model=AGENT_MODEL, tools=ALL_TOOLS, system_prompt=PROMPT, hooks=[hook], callback_handler=None)

for q in ["How much is a room at the Cliffside Resort?",
          "What's the weather forecast for Tokyo next week?",
          "Book Cliffside Resort for John Smith for 2 nights"]:
    r = await tool_agent.invoke_async(q)
    print(f"Q: {q}")
    print(f"   tool chosen: {hook.last[0]}  (confidence {hook.last[1]})")
    print(f"   answer:      {str(r)[:90]}\n")

## Option 2: choose the model

Strands' `ModelRouter` sends each request to one of several models; the choice is
a pluggable `RoutingStrategy`. The strategy here asks the Decisions API which
model fits, judging the request against each candidate's description. The router
is passed as the agent's model (Strands' seam for picking the model).

In [ ]:
from typing import Any
from strands.models import ModelRouter, RoutingCandidate, RoutingContext, RoutingStrategy

class OpenAIRoutingStrategy(RoutingStrategy):
    def __init__(self):
        self.decisions = []
    async def select(self, context: RoutingContext, **kwargs: Any) -> RoutingCandidate | None:
        if context.attempts:
            return None
        query = latest_user_text(context.messages)
        if not query:
            return None
        choices = [{"value": c.name, "description": c.description or c.name} for c in context.candidates]
        resp = await OAI.decisions.create(
            model="gpt-6-luna", input=query,
            questions=[{"type": "choice", "name": "model",
                        "instructions": "Which model should handle this request?", "choices": choices}],
        )
        answer = next(a for a in resp.answers if getattr(a, "name", None) == "model")
        self.decisions.append((answer.choice, answer.confidence))
        return next((c for c in context.candidates if c.name == answer.choice), None)

strategy = OpenAIRoutingStrategy()
router = ModelRouter(
    models=[
        RoutingCandidate(BedrockModel(model_id="us.amazon.nova-lite-v1:0", region_name=os.environ["AWS_REGION"]),
                         name="routine",  description="Short factual questions and routine lookups."),
        RoutingCandidate(BedrockModel(model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0", region_name=os.environ["AWS_REGION"]),
                         name="advanced", description="Multi-step reasoning, systems design, trade-off analysis."),
    ],
    strategy=strategy,
)
routed_agent = Agent(model=router, callback_handler=None)

QUESTIONS = [
    "What time zone is Tokyo in?",
    "What is the capital of Australia?",
    "Design a rollback-safe migration from regional to global idempotency keys.",
    "Compare optimistic vs pessimistic locking for a high-contention ledger and recommend one.",
]
for q in QUESTIONS:
    n = len(strategy.decisions)
    r = await routed_agent.invoke_async(q)
    chosen, conf = strategy.decisions[n]
    print(f"Q: {q}")
    print(f"   model chosen: {chosen}  (confidence {conf})")
    print(f"   answer:       {str(r)[:90]}\n")